In [ ]:
from ultralytics import YOLO
from pathlib import Path
# Load a COCO-pretrained YOLO12x model
model = YOLO("yolo12x.pt")

### Detekcia


In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

animal_classes = {'cow', 'sheep', 'bear', 'dog', 'cat', 'giraffe'}

path = r"C:\BP\pythonProject1\data_rysy\rys_trening_data_Beno\rys_trening_data_Beno"
image_paths = list(Path(path).rglob("*.jpg"))

images_with_animals = 0
processed_images = 0

for path in image_paths:
    if str(path).split('\\')[-1][0] < 'M':
        continue

    processed_images += 1

    results = model(str(path), conf=0.25)
    img = cv2.imread(str(path))

    if img is None:
        print(f"Could not load {path}")
        continue

    found_animal = False

    # Draw boxes
    for box in results[0].boxes:
        xyxy = box.xyxy[0].cpu().numpy().astype(int)
        cls_id = int(box.cls[0].item())
        label = model.names[cls_id]
        conf = box.conf[0].item()

        if label.lower() in animal_classes:
            found_animal = True

        cv2.rectangle(img, (xyxy[0], xyxy[1]), (xyxy[2], xyxy[3]), (0, 255, 0), 2)
        text_y = max(xyxy[1] - 10, 10)
        cv2.putText(
            img,
            f'{label} {conf:.2f}',
            (xyxy[0], text_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 255, 0),
            2
        )

    if found_animal:
        images_with_animals += 1

    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"{path.name}")
    plt.axis("off")
    plt.show()

print(f"\nProcessed images: {processed_images}")
print(f"Images with at least one target animal: {images_with_animals}")

### Ulozenie do suborovej struktury

In [ ]:
from pathlib import Path
import cv2
import os

# Define YOLO model and label you're interested in
animal_classes = {'cow', 'sheep', 'bear', 'dog', 'cat', 'giraffe'} 

# Paths
src_root = Path(r"C:\BP\pythonProject1\data_rysy\rys_trening_data_Beno\rys_trening_data_Beno")
dst_root = Path(r"C:\BP\pythonProject1\data_rysy\rys_trening_data_Beno\rys_trening_data_Beno_test")

# Get all image paths
image_paths = list(src_root.rglob("*.jpg"))

for image_path in image_paths:
    results = model(str(image_path))
    img = cv2.imread(str(image_path))

    if img is None:
        print(f"Could not load {image_path}")
        continue

    found = False
    for box in results[0].boxes:
        cls_id = int(box.cls[0].item())
        label = model.names[cls_id]

        if label.lower() in animal_classes:
            xyxy = box.xyxy[0].cpu().numpy().astype(int)
            x1, y1, x2, y2 = xyxy

            # Crop image
            cropped = img[y1:y2, x1:x2]

            # Define destination path
            rel_path = image_path.relative_to(src_root)
            save_dir = dst_root / rel_path.parent
            save_dir.mkdir(parents=True, exist_ok=True)
            save_path = save_dir / image_path.name

            # Save cropped image
            cv2.imwrite(str(save_path), cropped)
            found = True
            break  # Only crop/save once per image

    if not found:
        print(f"No lynx detected in {image_path}")
